# H3 · Communication calendars, activities and ideal battery storage

Use **Lunar H0 (Python 3.12)**. This notebook has no network or hardware calls.
Geometric contact windows, actual station booking/capacity, and radio-link quality
are different things. H3 connects the first two as explicit inputs; it does not model
link budgets. Battery examples use constant power, ideal efficiency and fixed reserves.
All examples below are synthetic teaching data, not DSN bookings or spacecraft models.

In [1]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
from html import escape
import os
import sys
from IPython.display import HTML, display
start = Path(os.environ.get("LUNAR_REPO_ROOT", Path.cwd())).resolve()
roots = [start, *start.parents, start / "quantum-hybrid-workspace", start / "h0-hybrid-workspace"]
ROOT = next((p for p in roots if (p / "backend/app/hybrid_resources.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open from the repository or set LUNAR_REPO_ROOT.")
sys.path.insert(0, str(ROOT))
from shared.hybrid import Scenario, Source, TimeWindow, Plan, load_scenario, fingerprint
from backend.app.hybrid_planning import solve_exact, validate_plan
from backend.app.hybrid_quantum import compile_qubo, run_qaoa
from backend.app.hybrid_resources import (
    ActivityLoad, BatteryProfile, GenerationSegment, StationCalendar,
    contact_from_r2, integrate_resources, battery_trace,
)
import backend.app.hybrid_resources as resource_module
assert Path(resource_module.__file__).resolve() == ROOT / "backend/app/hybrid_resources.py"
T0 = datetime(2026, 11, 4, tzinfo=timezone.utc)
def window(start_h, end_h):
    return TimeWindow(start_utc=T0+timedelta(hours=start_h), end_utc=T0+timedelta(hours=end_h))
def evidence(name, description):
    return Source(source_id=name, kind="synthetic", description=description)
def table(headers, rows):
    def row(values, tag):
        return "<tr>"+"".join(f"<{tag}>{escape(str(v))}</{tag}>" for v in values)+"</tr>"
    display(HTML("<table>"+row(headers,"th")+"".join(row(r,"td") for r in rows)+"</table>"))
raw = load_scenario(ROOT / "shared/examples/hybrid/independent.json").model_dump(mode="json")
raw["resources"] = []
for candidate in raw["candidates"]:
    candidate["demands"] = []
base = Scenario.model_validate(raw)
print("Base alternatives:", len(base.candidates), "all resource assumptions will be explicit.")

Base alternatives: 12 all resource assumptions will be explicit.


## 1. Adapt the real R2 result shape without inventing a booking

The following dictionary is explicitly synthetic but uses R2's `stations -> intervals`
shape. Tests also call the actual `communication_windows` function and adapt its output.
To use a saved real result, supply its station intervals and traceable source metadata;
check that candidate dates/sites correspond to that analysis. Do not relabel these fixtures
as NASA observations.

In [2]:
r2_fixture = {"stations": {"DSS-14": {"intervals": [
    dict(window(0,12).model_dump(mode="json"), state="available", duration_s=43200)
]}}}
geometry_source = evidence("h3-notebook-geometry", "Synthetic R2-shaped all-day geometry for teaching only.")
contacts = tuple(contact_from_r2(r2_fixture, requirement_id="contact-"+c.option_id,
    option_id=c.option_id, station_id="DSS-14", window=c.window, source=geometry_source)
    for c in base.candidates)
calendar = StationCalendar(station_id="DSS-14", capacity_channels=1,
    availability=(window(0,12),), source=evidence("h3-notebook-calendar", "Assumed one booked channel; not a DSN allocation."))
contact_problem = integrate_resources(base, stations=(calendar,), contacts=contacts)
contact_solution = solve_exact(contact_problem)
assert contact_solution.feasible_count == 24
assert contact_solution.validation.exact_score == "13/5"
print("Geometric contacts plus shared booking:", contact_solution.status)
print("Resources:", len(contact_problem.resources), "= one shared calendar + twelve candidate-specific geometry gates")
print("Best score:", contact_solution.validation.score)

Geometric contacts plus shared booking: optimal
Resources: 13 = one shared calendar + twelve candidate-specific geometry gates
Best score: 2.6


## 2. Visible does not mean booked; unknown does not mean available

A closed calendar blocks use even if the Moon/Earth geometry is available. An unknown
geometric interval overlapping a required contact creates incomplete input. Unknown time
outside the required contact does not block an otherwise complete option.

In [3]:
closed = integrate_resources(base, stations=(calendar.model_copy(update={"availability": ()}),), contacts=contacts)
assert solve_exact(closed).status == "infeasible"
unknown_fixture = {"stations": {"DSS-14": {"intervals": [dict(window(0,12).model_dump(mode="json"), state="unknown")]}}}
unknown_contacts = tuple(contact_from_r2(unknown_fixture, requirement_id="contact-"+c.option_id,
    option_id=c.option_id, station_id="DSS-14", window=c.window, source=geometry_source) for c in base.candidates)
unknown = integrate_resources(base, stations=(calendar,), contacts=unknown_contacts)
assert solve_exact(unknown).status == "incomplete"
table(["Input", "Classical result"], [["Visible + booked", contact_solution.status],
    ["Visible + closed calendar", solve_exact(closed).status], ["Unknown geometry + booked", solve_exact(unknown).status]])

Input,Classical result
Visible + booked,optimal
Visible + closed calendar,infeasible
Unknown geometry + booked,incomplete


## 3. Add an energy constraint that actually changes the selected options

Give each mission its own ideal 3600 J battery with no recharge. The highest-score
option consumes 2 W for an hour (7200 J); the remaining options consume 1 W (3600 J).
These toy powers are explicit assumptions. The first option is now infeasible for each
mission, leaving the three lower-score slots to share one station channel.

In [4]:
batteries = tuple(BatteryProfile(battery_id="battery-"+m.mission_id,
    initial_energy_j=3600, capacity_j=3600, reserve_j=0,
    source=evidence("source-battery-"+m.mission_id, "Synthetic 3600 J ideal battery; no recharge."))
    for m in base.missions)
loads = tuple(ActivityLoad(activity_id="activity-"+c.option_id, option_id=c.option_id,
    battery_id="battery-"+c.mission_id, window=c.window,
    power_w=2 if c.option_id.endswith("-1") else 1,
    source=evidence("h3-notebook-loads", "Synthetic option power: 2 W for option 1, otherwise 1 W."))
    for c in base.candidates)
integrated = integrate_resources(base, stations=(calendar,), contacts=contacts, batteries=batteries, activities=loads)
classical = solve_exact(integrated)
assert classical.status == "optimal" and classical.feasible_count == 6
assert classical.validation.exact_score == "11/5"
assert all(not oid.endswith("-1") for oid in classical.plan.selected_option_ids)
print("Integrated input fingerprint:", fingerprint(integrated))
print("Feasible plans:", classical.feasible_count, "/", classical.total_combinations)
print("New optimum:", classical.validation.score, "raw science:", classical.validation.metric_totals["science_value"])
print("Selected:", classical.plan.selected_option_ids)

Integrated input fingerprint: 4f28292c4c7519d2f96f2d75943c5c5ac03626fa8d900fd1257f90fff33b2f62
Feasible plans: 6 / 64
New optimum: 2.2 raw science: 22.0
Selected: ('mission-1-option-2', 'mission-2-option-3', 'mission-3-option-4')


## 4. Run quantum and classical selection on exactly the same integrated input

The same candidate IDs, source metadata, deadlines, station demands and storage constraints
reach both solvers. H2 proves representability before running. This small example remains
unary/pairwise after encoding and fits six qubits. Final samples are checked by H1.

In [5]:
quantum = run_qaoa(integrated, depth=1, evaluations=80, shots=256, seed=17)
assert quantum.input_fingerprint == classical.input_fingerprint
assert abs(quantum.uniform_feasible_probability - 6/64) < 1e-12
if quantum.best_sampled_plan:
    assert validate_plan(quantum.best_sampled_plan, integrated).status == "feasible"
table(["Method", "Result", "Best score", "Elapsed s"], [
    ["Classical exact", classical.status, classical.validation.score, classical.elapsed_s],
    ["Local QAOA", "raw samples; no repair", validate_plan(quantum.best_sampled_plan, integrated).score if quantum.best_sampled_plan else None, quantum.elapsed_s],
])
print("Uniform feasible/optimal probability:", quantum.uniform_feasible_probability)
print("QAOA feasible probability:", quantum.feasible_probability)
print("QAOA optimal probability:", quantum.optimal_probability)
print("Observed feasible rate:", quantum.sampled_feasible_rate)
print("Qubits:", quantum.logical_qubits, "hardware jobs:", quantum.hardware_jobs)

Method,Result,Best score,Elapsed s
Classical exact,optimal,2.2,0.017969041000469588
Local QAOA,raw samples; no repair,2.2,4.0478512499976205


Uniform feasible/optimal probability: 0.09375
QAOA feasible probability: 0.409740960022292
QAOA optimal probability: 0.409740960022292
Observed feasible rate: 0.40234375
Qubits: 6 hardware jobs: 0


## 5. Inspect the energy trace of a selected plan

An independent propagation adds supplied generation, subtracts selected activity consumption
and spills surplus above capacity. It reports a deficit without silently clipping it to zero.
Use the same explicit battery/load inputs used for integration; the trace is a diagnostic of
those supplied profiles, not a separate physical certificate or a profile-identity verifier.

In [6]:
for profile in batteries:
    trace = battery_trace(classical.plan, integrated, profile, loads)
    assert not any(row.reserve_violated for row in trace)
    print(profile.battery_id)
    table(["Start UTC", "End UTC", "Start J", "Generated J", "Used J", "Spilled J", "End J", "Reserve violated"], [
        [r.window.start_utc.isoformat(), r.window.end_utc.isoformat(), r.start_energy_j, r.generated_energy_j,
         r.consumed_energy_j, r.spilled_energy_j, r.end_energy_j, r.reserve_violated] for r in trace
    ])

battery-mission-1


Start UTC,End UTC,Start J,Generated J,Used J,Spilled J,End J,Reserve violated
2026-11-04T00:00:00+00:00,2026-11-04T01:00:00+00:00,3600.0,0.0,0.0,0.0,3600.0,False
2026-11-04T01:00:00+00:00,2026-11-04T02:00:00+00:00,3600.0,0.0,3600.0,0.0,0.0,False
2026-11-04T02:00:00+00:00,2026-11-04T12:00:00+00:00,0.0,0.0,0.0,0.0,0.0,False


battery-mission-2


Start UTC,End UTC,Start J,Generated J,Used J,Spilled J,End J,Reserve violated
2026-11-04T00:00:00+00:00,2026-11-04T02:00:00+00:00,3600.0,0.0,0.0,0.0,3600.0,False
2026-11-04T02:00:00+00:00,2026-11-04T03:00:00+00:00,3600.0,0.0,3600.0,0.0,0.0,False
2026-11-04T03:00:00+00:00,2026-11-04T12:00:00+00:00,0.0,0.0,0.0,0.0,0.0,False


battery-mission-3


Start UTC,End UTC,Start J,Generated J,Used J,Spilled J,End J,Reserve violated
2026-11-04T00:00:00+00:00,2026-11-04T03:00:00+00:00,3600.0,0.0,0.0,0.0,3600.0,False
2026-11-04T03:00:00+00:00,2026-11-04T04:00:00+00:00,3600.0,0.0,3600.0,0.0,0.0,False
2026-11-04T04:00:00+00:00,2026-11-04T12:00:00+00:00,0.0,0.0,0.0,0.0,0.0,False


## 6. Show why total generated energy alone is not enough

A shared battery starts full at 7200 J, and generation supplies 1 W during the first hour.
Each selected activity consumes 1 W for one hour. If all activities start later, the initial
3600 J production spills from the full battery: only 7200 J remains for 10800 J consumption.
At least one mission must consume during that first hour. There are 37 feasible assignments.

In [7]:
shared_battery = BatteryProfile(battery_id="shared-storage", initial_energy_j=7200, capacity_j=7200, reserve_j=0,
    generation=(GenerationSegment(window=window(0,1), power_w=1,
        source=evidence("h3-notebook-generation", "Assumed 1 W for the first hour; not derived from illumination.")),),
    source=evidence("h3-notebook-shared-battery", "Synthetic initially full 7200 J shared battery."))
shared_loads = tuple(ActivityLoad(activity_id="shared-load-"+c.option_id, option_id=c.option_id,
    battery_id=shared_battery.battery_id, window=c.window, power_w=1,
    source=evidence("h3-notebook-shared-loads", "Assumed 1 W per selected mission.")) for c in base.candidates)
spillage_problem = integrate_resources(base, batteries=(shared_battery,), activities=shared_loads)
spillage_result = solve_exact(spillage_problem)
assert spillage_result.feasible_count == 37
late = Plan(plan_id="late-demo", scenario_id=spillage_problem.scenario_id, scenario_revision=spillage_problem.revision,
    input_fingerprint=fingerprint(spillage_problem), state_revisions={},
    selected_option_ids=tuple(f"mission-{i}-option-2" for i in (1,2,3)), execution_mode="classical", solver_name="manual-example")
assert validate_plan(late, spillage_problem).status == "infeasible"
late_trace = battery_trace(late, spillage_problem, shared_battery, shared_loads)
table(["Start UTC", "Generated J", "Used J", "Spilled J", "End J", "Reserve violated"], [
    [r.window.start_utc.isoformat(), r.generated_energy_j, r.consumed_energy_j, r.spilled_energy_j,
     r.end_energy_j, r.reserve_violated] for r in late_trace
])
print("Feasible assignments:", spillage_result.feasible_count)

Start UTC,Generated J,Used J,Spilled J,End J,Reserve violated
2026-11-04T00:00:00+00:00,3600.0,0.0,3600.0,7200.0,False
2026-11-04T01:00:00+00:00,0.0,10800.0,0.0,-3600.0,True
2026-11-04T02:00:00+00:00,0.0,0.0,0.0,-3600.0,True


Feasible assignments: 37


## 7. Keep the classical answer when the bounded quantum mapping is unsupported

The spillage problem has a genuine three-way restriction. H2's current no-slack encoding
cannot express it exactly using only unary/pair constraints. We retain the classical answer
and report the quantum limit; we do not delete the energy rule or change the input to make
a quantum comparison look successful.

In [8]:
try:
    compile_qubo(spillage_problem)
except ValueError as error:
    print("Quantum path unsupported:", error)
    print("Classical answer remains available:", spillage_result.status, spillage_result.validation.score)
else:
    raise AssertionError("Higher-order storage rule was silently approximated")
print("All examples remain synthetic. No DSN allocation, radio-link guarantee or battery hardware certification.")
print("Hardware jobs submitted:", quantum.hardware_jobs)

Quantum path unsupported: Resource constraint requires higher-order or slack-variable mapping
Classical answer remains available: optimal 3.0
All examples remain synthetic. No DSN allocation, radio-link guarantee or battery hardware certification.
Hardware jobs submitted: 0
